[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Why SQLAlchemy &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: the engine, the `students` table, the `Student` class, `search_students` and
`show_sql`. Run it first. The tasks do not depend on one another, and the last cell removes the
scratch folder.


In [1]:
import shutil
import sqlite3
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import Column, Date, Integer, MetaData, String, Table, create_engine, select, text
from sqlalchemy.dialects import mssql, postgresql
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
STARTS = ["2024-08-26", "2025-01-13", "2025-08-25"]           # the first day of three terms
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], STARTS[i % len(STARTS)]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]

build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL);
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL);
""")
build.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)", STUDENTS)
build.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
build.commit()
build.close()


def show_sql(statement, dialect):
    """Print the SQL a statement becomes for one database, and the values that travel beside it."""
    compiled = statement.compile(dialect=dialect)
    for line in str(compiled).splitlines():
        print("   ", line.rstrip())
    print("    values:", compiled.params)


engine = create_engine(f"sqlite:///{DATABASE}")


class Base(DeclarativeBase):
    pass


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    email: Mapped[str]
    program: Mapped[str]
    started_on: Mapped[date]

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


def search_students(session, program=None, started_after=None, name=None, limit=10):
    """The registrar's search: the students matching every box that was filled in, by name."""
    query = select(Student)
    if program:
        query = query.where(Student.program == program)
    if started_after:
        query = query.where(Student.started_on >= started_after)
    if name:
        query = query.where(Student.name.contains(name))
    return session.scalars(query.order_by(Student.name).limit(limit)).all()


print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "holds", len(STUDENTS), "students")


sqlalchemy 2.0.54 | scratch/college.db holds 25 students


**1.** History students who started on or after 1 January 2025.


In [2]:
with Session(engine) as session:
    for student in search_students(session, program="History", started_after=date(2025, 1, 1)):
        print(student, "started", student.started_on)


Student('Elena Petrova', 'History') started 2025-01-13
Student('Olivia Brandt', 'History') started 2025-08-25
Student('Tara Nilsen', 'History') started 2025-01-13


Three of the five History students started in 2025, and the first day went in as a `date`, which
the `Date` column compares with what it stores.


**2.** The SQL of a search with every box filled in, for SQLite.


In [3]:
FULL_SEARCH = (
    select(Student)
    .where(Student.program == "History")
    .where(Student.started_on >= date(2025, 1, 1))
    .where(Student.name.contains("O'"))
    .order_by(Student.name)
)
show_sql(FULL_SEARCH, engine.dialect)


    SELECT students.id, students.name, students.email, students.program, students.started_on
    FROM students
    WHERE students.program = ? AND students.started_on >= ? AND (students.name LIKE '%' || ? || '%') ORDER BY students.name
    values: {'program_1': 'History', 'started_on_1': datetime.date(2025, 1, 1), 'name_1': "O'"}


Three conditions joined by `AND`, a `?` for every value, and the values beside the SQL: the program,
the date, still a `date` until the statement runs and the `Date` type turns it into the text SQLite
stores, and the name. `select(Student)` selects every column the class maps.


**3.** The same statement for PostgreSQL.


In [4]:
show_sql(FULL_SEARCH, postgresql.dialect())


    SELECT students.id, students.name, students.email, students.program, students.started_on
    FROM students
    WHERE students.program = %(program_1)s AND students.started_on >= %(started_on_1)s AND (students.name LIKE '%%' || %(name_1)s || '%%') ORDER BY students.name
    values: {'program_1': 'History', 'started_on_1': datetime.date(2025, 1, 1), 'name_1': "O'"}


PostgreSQL's driver takes named placeholders in the form `%(name)s`, one for every value, with the
same names as the values. A `%` that belongs to the SQL itself is doubled, `'%%'`, so that the driver
does not take it for the start of a placeholder. The date is still a `date`, since PostgreSQL has a
date type of its own and the driver sends one.


**4.** Students per first day, with `text()`.


In [5]:
PER_START = text("""
    SELECT started_on, COUNT(*) AS students
    FROM students
    GROUP BY started_on
    ORDER BY started_on
""")

with engine.connect() as conn:
    for row in conn.execute(PER_START):
        print(repr(row.started_on), row.students)


'2024-08-26' 9
'2025-01-13' 8
'2025-08-25' 8


`text()` needs no bind parameters when there are no values to send. Its rows still answer to column
names, `row.students` for the column the SQL named with `AS`. `started_on` comes back as text here:
`text()` knows nothing about the column's type, where a `Table` or a mapped class does.


**5.** Pages of results.


In [6]:
def search_students(session, program=None, started_after=None, name=None, limit=10, page=1):
    """The registrar's search, one page of results at a time, counting pages from 1."""
    query = select(Student)
    if program:
        query = query.where(Student.program == program)
    if started_after:
        query = query.where(Student.started_on >= started_after)
    if name:
        query = query.where(Student.name.contains(name))
    return session.scalars(query.order_by(Student.name).limit(limit).offset((page - 1) * limit)).all()


with Session(engine) as session:
    print("page 1:", search_students(session, limit=5))
    print("page 2:", search_students(session, limit=5, page=2))


page 1: [Student('Ana Reyes', 'Biology'), Student("Aoife O'Brien", 'History'), Student('Ben Okafor', 'Computer Science'), Student('Chloe Martin', 'Mathematics'), Student('Daniel Kim', 'Psychology')]
page 2: [Student('Elena Petrova', 'History'), Student('Felix Wagner', 'Biology'), Student('Grace Lin', 'Computer Science'), Student('Hassan Ali', 'Mathematics'), Student('Isabel Costa', 'Psychology')]


`offset` skips the rows of the pages before, and the order by name keeps every page in the same order
from one request to the next, which paging depends on: without an `ORDER BY`, the database may
return rows in any order it likes.


**6.** A `Course` class.


In [7]:
class Course(Base):
    __tablename__ = "courses"

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str]
    title: Mapped[str]
    department: Mapped[str]
    credits: Mapped[int]


with Session(engine) as session:
    for course in session.scalars(select(Course).where(Course.credits == 4).order_by(Course.code)):
        print(course.code, course.title, course.credits)


BIO-101 Introduction to Biology 4
CHE-110 General Chemistry 4
MAT-120 Calculus I 4
MAT-121 Calculus II 4


`Course` shares `Base` with `Student`, so the two classes belong to the same family, and the same
session runs statements for both. The four 4-credit courses are the sciences and the two calculus
courses.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Why SQLAlchemy](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/01-why-sqlalchemy.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
